# Mathematical Foundation: Continuous to Discrete Dynamics

This notebook details the mathematical derivation for the neuron-astrocyte network model. We begin with the continuous-time Ordinary Differential Equations (ODEs) that describe the biological dynamics of the tripartite synapse, as presented in the original PLOS paper. 
Our objective is to transform these continuous models into discrete-time update rules that can be efficiently computed as a computational graph in PyTorch.



$$ \begin{align*}
\tau \dot{x} &= -x + W \phi (x) \\
\tau \dot{W} &= -W + C \Phi (x) + D \psi (z) \\
\dot{z} &= -z + F\psi(z) + H \Phi(x)
\end{align*}
$$

where:
- $x = [x_1, \dots, x_n]^T$
- $z = [z_1, \dots, z_m]^T$
- $\phi(x) = [\phi(x_1), \dots, \phi(x_n)]^T$ and $\psi(x) = [\psi(x_1), \dots, \psi(x_n)]^T$ 


### Forward Euler Discretization
To simulate this system computationally, we cannot use continuous time. We must discretize the ODEs by applying the **Forward Euler method**. We approximate the continuous derivative $\frac{dx}{dt}$ as a finite difference over a discrete time step $\Delta t$.


$$
\begin{align*}
\frac{\Delta x}{\Delta t} &= -x_{t-1} + W_{t-1} \phi (x_{t-1}) \\
\frac{\Delta W}{\Delta t} &= -W_{t-1} + C \Phi (x_{t-1}) + D \psi (z_{t-1}) \\
\frac{\Delta z}{\Delta t} &= \tau(-z_{t-1} + F\psi(z_{t-1}) + H \Phi(x_{t-1}))
\end{align*}
$$

<br>

$$
\begin{align*}
\Delta x &= ( -x_{t-1} + W_{t-1} \phi (x_{t-1}) )  \Delta t \\
\Delta W &= ( -W_{t-1} + C \Phi (x_{t-1}) + D \psi (z_{t-1}) )  \Delta t \\
\Delta z &= \tau( -z_{t-1} + F\psi(z_{t-1}) + H \Phi(x_{t-1}) )  \Delta t
\end{align*}
$$

<br>

$$
\begin{align*}
 x_t - x_{t-1} &= ( -x_{t-1} + W_{t-1} \phi (x_{t-1}) )  \Delta t \\
 W_t - W_{t-1} &= ( -W_{t-1} + C \Phi (x_{t-1}) + D \psi (z_{t-1}) )  \Delta t \\
 z_t - z_{t-1} &= \tau( -z_{t-1} + F\psi(z_{t-1}) + H \Phi(x_{t-1}) )  \Delta t
\end{align*}
$$

<br>

$$
\begin{align*}
 x_t  &= ( -x_{t-1} + W_{t-1} \phi (x_{t-1}) )  \Delta t +  x_{t-1} \\
 W_t  &= ( -W_{t-1} + C \Phi (x_{t-1}) + D \psi (z_{t-1}) )  \Delta t +  W_{t-1} \\
 z_t  &= \tau( -z_{t-1} + F\psi(z_{t-1}) + H \Phi(x_{t-1}) )  \Delta t +  z_{t-1} 
\end{align*}
$$

<br>

$$
\begin{align*}
x_t  &= ( -x_{t-1} + W_{t-1} \phi (x_{t-1}) )  \Delta t +  x_{t-1} \\
W_t  &= ( -W_{t-1} + C \Phi (x_{t-1}) + D \psi (z_{t-1}) )  \Delta t +  W_{t-1} \\
z_t  &= \tau( -z_{t-1} + F\psi(z_{t-1}) + H \Phi(x_{t-1}) )  \Delta t +  z_{t-1} 
\end{align*}
$$

<br>

$$
\begin{align*}
x_t  &=  -x_{t-1}\Delta t  + W_{t-1} \phi (x_{t-1})\Delta t  +  x_{t-1} \\
W_t  &=  -W_{t-1} \Delta t  + C \Phi (x_{t-1}) \Delta t  + D \psi (z_{t-1}) \Delta t  +  W_{t-1} \\
z_t  &=  -z_{t-1} \tau \Delta t  + F\psi(z_{t-1}) \tau \Delta t  + H \Phi(x_{t-1}) \tau  \Delta t +  z_{t-1} 
\end{align*}
$$

<br>

$$
\begin{align*}
x_t  &=  (1 - \Delta t)  x_{t-1}   + W_{t-1} \phi (x_{t-1})\Delta t \\
W_t  &=  (1- \Delta t) W_{t-1}  + C \Phi (x_{t-1}) \Delta t  + D \psi (z_{t-1}) \Delta t  \\
z_t  &=  (1 - \tau \Delta t) z_{t-1}  + F\psi(z_{t-1}) \tau \Delta t  + H \Phi(x_{t-1}) \tau  \Delta t 
\end{align*}
$$


### Substitution and Biological Intuition
We substitute the time step $\Delta t$ with the parameter $\gamma$, which acts similarly to a learning rate or integration step.

In this discrete form, the equations reveal an intuitive mechanism:
* The term $(1 - \gamma)$ acts as a **leakage factor** (decay). Without new inputs, the neuron's potential naturally decays over time.
* The $\gamma$ multiplier scales the impact of incoming synaptic signals.
* Importantly, the astrocyte equation ($z_t$) includes the $\tau$ parameter ($\tau \ll 1$). The resulting $\tau \gamma$ multiplier ensures that the astrocytic state updates **significantly slower** than the neuronal state, providing the required time-scale separation for contextual memory.


$$
\begin{align*}
x_t  &=  (1 - \gamma)  x_{t-1}   + W_{t-1} \phi (x_{t-1})\gamma \\
W_t  &=  (1- \gamma) W_{t-1}  + C \Phi (x_{t-1}) \gamma  + D \psi (z_{t-1}) \gamma  \\
z_t  &=  (1 - \tau \gamma) z_{t-1}  + F\psi(z_{t-1}) \tau \gamma  + H \Phi(x_{t-1}) \tau \gamma
\end{align*}
$$

<br><br>
adding inputs and output terms

$$
\begin{align*}
x_t  &=  (1 - \gamma)  x_{t-1} + \gamma (W_{t-1} \phi (x_{t-1}) + W_\text{in}^1 I_t) \\
W_t  &=  (1- \gamma) W_{t-1}  + \gamma( C \Phi (x_{t-1})+ D \psi (z_{t-1}) )   \\
z_t  &=  (1 - \tau \gamma) z_{t-1}  + \gamma  \tau (F\psi(z_{t-1}) + H \Phi(x_{t-1}) + W_\text{in}^2 I_t) \\
y_t &= W_{\text{out}} x_t + b_{\text{out}}
\end{align*}
$$

<br>



### Architectural Bottleneck: The $O(n^4)$ Memory Problem

The derivation above highlights a critical engineering constraint. The outer product $\Phi(x)$ produces a vector of size $n^2$. Consequently, the matrix $C$, which maps this relation, must have dimensions $n^2 \times n^2$. 

For a network of just $n=128$ neurons, the $C$ matrix would contain over 268 million parameters. Calculating $C \Phi(x)$ using standard matrix multiplication would allocate gigabytes of VRAM per step. **This mathematical insight dictates that in the actual PyTorch implementation, we must avoid explicit diagonal matrices and instead use memory-efficient element-wise Hadamard products (e.g., `C.unsqueeze(1) * X`).**


# Matrices Dimensions

## Tensor Shape Derivations for PyTorch

To implement the derived discrete equations using hardware-accelerated tensor operations (avoiding inefficient `for` loops), we must strictly define the dimensionalities of all weight matrices and state vectors.

$$
x \in \mathbb{R}^{n} \implies (W \phi(x) + W_\text{in}^1 I) \in \mathbb{R}^{n}
$$


$$
\phi(x) \in \mathbb{R}^{n} \implies W \in \mathbb{R}^{n \times n} 
$$

$$
I \in \mathbb{R}^{|u|} \implies W \in \mathbb{R}^{n \times |u|} 
$$



<br><br><br>

$$
W \in \mathbb{R}^{n \times n} \implies ( C \Phi(x) + D \phi(z) ) \in \mathbb{R}^{n \times n} 
$$

$$
\Phi(x) \in \mathbb{R}^{n^2} \implies C \in \mathbb{R}^{n^2 \times n^2}
$$

$$
\psi(x) \in \mathbb{R}^{m} \implies D \in \mathbb{R}^{n^2 \times m}
$$


$$
(C \Phi(x) \in \mathbb{R}^{n^2} \land D \phi(z) \in \mathbb{R}^{n^2}) \implies \text{reshape}(\mathbb{R}^{n^2}) \rightarrow \mathbb{R}^{n \times n} 
$$

<br><br><br>


$$
z \in \mathbb{R}^{m} \implies ( F \psi(z) + H \Phi(x) + W_\text{in}^2 I) \in \mathbb{R}^{m} 
$$


$$
\psi(z) \in \mathbb{R}^{m} \implies F \in  \mathbb{R}^{m \times m}
$$

$$
\Phi(x) \in \mathbb{R}^{n^2} \implies H \in  \mathbb{R}^{m \times n^2}
$$

$$
I \in \mathbb{R}^{|u|} \implies W_\text{in}^2 \in \mathbb{R}^{m \times |u|}
$$

<br><br><br>

$$ 
y \in \mathbb{R}^{|y|} \implies (W_\text{out} x_t + b_\text{out}) \in \mathbb{R}^{|y|} 
$$

$$
x_t \in \mathbb{R}^{n} \implies W_\text{out} \in \mathbb{R}^{|y| \times n}
$$

$$
y \in \mathbb{R}^{|y|} \implies b_\text{out} \in \mathbb{R}^{|y|}
$$

# Multi-Armed Bandits



To evaluate the network's ability to act as a contextual switchboard, we deploy it in a Reinforcement Learning paradigm. The agent interacts with a $k$-armed Bernoulli Bandit. The objective is to maximize the expected cumulative reward by learning the underlying, hidden reward probabilities ($\mu_i$) of each arm.

$$\max_{a_t \in \mathcal{A}} \sum_{t=1}^T r_t$$

<br>

$$\mathcal{A} = \{a_1, a_2, \dots, a_n \}$$

$$r_i = \text{reward}({a_i}) \sim \text{Bernoulli}(\mu_i), \ \ \ \  i = 1, \dots, n$$

$$\mu_i = \mathbb{E}\big[r_i]$$

<br><br>

### The Evaluation Metric: Cumulative Regret
Rather than tracking raw rewards, we evaluate the model using **Cumulative Regret ($R_T$)**. Regret quantifies the "cost of ignorance"—the difference in expected reward between the optimal arm and the arm the agent actually chose at time $t$.

$$R_T = \sum_{t=1}^T (\max_{a_i \in \mathcal{A}}{\mu_i} - \mathbb{E}[r_t])$$

* A linearly increasing regret indicates the agent is stuck in a local minimum (premature convergence).
* A flattening regret curve indicates the agent has successfully learned the optimal strategy and stopped exploring sub-optimal arms.